# 🐘 Apache Ozone Deep-Dive: Architettura, Stress Test & API S3

Questo laboratorio esplora in modo completo e interattivo **Apache Ozone (v2.2.1)**, il sistema di object storage distribuito di nuova generazione progettato per superare i limiti architetturali di Apache HDFS.

---

### 🎯 Contenuti del Laboratorio
1. **La Nuova Gerarchia**: Volume ➔ Bucket ➔ Key (confronto con il filesystem piatto/ad albero di HDFS)
2. **Lo "Small File Stress Test" (La Killer Feature ⭐⭐⭐)**: Dimostrazione del perché Ozone risolve il problema storico dei piccoli file grazie a **RocksDB** nell'Ozone Manager (OM)
3. **Compatibilità Nativa con l'API S3**: Interazione con l'S3 Gateway tramite **Boto3** e mapping bidirezionale con i volumi Ozone
4. **Monitoraggio con Ozone Recon**: Visualizzazione grafica dello stato del cluster su http://localhost:9888

---

# PARTE 1: 🏛️ La Nuova Gerarchia: Volume, Bucket, Key

In Hadoop HDFS classico, il filesystem è un albero monolitico (`/cartella/sottocartella/file`). Tutti i metadati risiedono nella **RAM del NameNode**.

Apache Ozone introduce una **gerarchia a tre livelli disaccoppiata e multi-tenant**:

```
Volume (Amministrazione, Quota di Spazio e Namespace, Utente Proprietario)
 └── Bucket (Contenitore logico, compatibile con i bucket S3)
      └── Key (L'oggetto o file memorizzato con i relativi blocchi dati)
```

- **Volume**: Creato generalmente dagli amministratori. Rappresenta un dipartimento, un progetto o un tenant (es. `/finance`, `/marketing`, `/volume-lab`). Qui si impostano le quote di storage e i permessi di accesso.
- **Bucket**: Equivalente a un bucket Amazon S3. Può appartenere a un solo volume. Gli utenti possono creare e gestire liberamente i propri bucket all'interno del proprio volume.
- **Key**: È l'oggetto dati vero e proprio (corrisponde al file o all'oggetto S3).

Possiamo interagire con questa gerarchia direttamente tramite la CLI di Ozone (`ozone sh`).

In [ ]:
# 1. Creiamo un Volume amministrativo per il nostro laboratorio
!ozone sh volume create /volume-lab

# Visualizziamo le informazioni del volume appena creato
!ozone sh volume info /volume-lab

In [ ]:
# 2. All'interno del Volume creiamo un Bucket dedicato ai dati
!ozone sh bucket create /volume-lab/dati

# Elenchiamo i bucket presenti nel volume
!ozone sh bucket list /volume-lab

In [ ]:
# 3. Creiamo un file di prova locale e carichiamolo come Key nel bucket
!echo 'Benvenuti in Apache Ozone 2.2.1!' > file_benvenuto.txt
!ozone sh key put /volume-lab/dati/file_benvenuto.txt file_benvenuto.txt

# Visualizziamo la lista delle chiavi nel bucket
!ozone sh key list /volume-lab/dati

In [ ]:
# 4. Leggiamo il contenuto della chiave direttamente da Ozone
!ozone sh key cat /volume-lab/dati/file_benvenuto.txt
!rm -f file_benvenuto.txt

---
# PARTE 2: ⭐⭐⭐ Lo "Small File Stress Test" (La Killer Feature)

### 💥 Il Problema Storico di HDFS: Lo "Small Files Problem"
In Apache HDFS, il **NameNode** conserva l'intero albero del filesystem e la posizione di tutti i blocchi **nella propria memoria RAM**. Ciascun file, cartella o blocco occupa circa **150 byte di RAM**.
- Se carichi 10.000 file piccoli da 1 KB ciascuno, consumi appena 10 MB di disco, ma crei 10.000 record nel NameNode.
- Su scala Big Data reale (es. milioni o centinaia di milioni di file generati da streaming Kafka, sensori IoT o micro-partizioni), il NameNode esaurisce la RAM, il Garbage Collector di Java va in stallo e l'intero cluster HDFS collassa.

### 🚀 La Rivoluzione di Apache Ozone: RocksDB nell'Ozone Manager (OM)
In Apache Ozone:
1. **Disaccoppiamento dei Metadati**: L'Ozone Manager (OM) gestisce lo spazio logico dei nomi (volumi, bucket, chiavi), mentre lo Storage Container Manager (SCM) gestisce i container di blocchi fisici.
2. **Storage Engine su Disco con Cache LRU**: L'Ozone Manager memorizza i metadati in un database incorporato ultra-veloce basato su Log-Structured Merge-Tree: **RocksDB**.
3. **Scalabilità a Miliardi di File**: Poiché i metadati risiedono su disco/SSD veloce e la RAM serve solo da cache, Ozone può gestire tranquillamente **più di 10 miliardi di oggetti/file** senza alcun collasso di memoria.

Questo è il vero momento "Aha!" di Apache Ozone.

---
### 🧪 Esecuzione del Test: Generazione e Caricamento Massivo di File Minuscoli

Possiamo generare e caricare migliaia di file da 1 KB. Mostriamo sia la sintassi da CLI bash che un benchmark multi-thread ad alte prestazioni in Python.

#### Approccio Bash CLI (Concettuale / Didattico)
Come indicato nella guida del laboratorio, il ciclo bash per caricare 10.000 file tramite la CLI è:
```bash
# Ciclo sequenziale bash:
for i in {1..10000}; do
    echo "dato $i" > file_$i.txt
    ozone sh key put /volume-lab/dati/file_$i.txt file_$i.txt
    rm -f file_$i.txt
done

# Conteggio degli oggetti risultanti:
ozone sh key list /volume-lab/dati | grep -o '"name"' | wc -l
```
> *Nota: Il ciclo sequenziale bash avvia una JVM per ogni singolo comando `ozone sh`, richiedendo diversi minuti. Eseguiamo invece il test ad alte prestazioni in parallelo con Python e Boto3 per caricare migliaia di oggetti in pochi secondi!*

In [ ]:
import os
import time
import boto3
from concurrent.futures import ThreadPoolExecutor
from botocore.client import Config

# Configurazione del test di stress
NUM_FILE_STRESS = 2000  # Puoi incrementare a 5000 o 10000 a piacere!
PAYLOAD_1KB = b"X" * 1024  # 1 Kilobyte esatto di payload per file
STRESS_BUCKET = "stress-test-small-files"

endpoint_url = os.environ.get("OZONE_S3_ENDPOINT", "http://s3g:9878")
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint_url,
    aws_access_key_id="your-access-key",
    aws_secret_access_key="your-secret-key",
    region_name="us-east-1",
    config=Config(s3={"addressing_style": "path"})
)

# Creiamo il bucket dedicato allo stress test
try:
    s3.create_bucket(Bucket=STRESS_BUCKET)
    print(f"✅ Bucket '{STRESS_BUCKET}' pronto per lo stress test.")
except Exception as e:
    print(f"Bucket già esistente o nota: {e}")

print(f"🚀 Inizio caricamento massivo di {NUM_FILE_STRESS} file minuscoli (1 KB ciascuno) su Ozone...")

def upload_worker(idx):
    key_name = f"small_files/file_{idx:05d}.txt"
    s3.put_object(Bucket=STRESS_BUCKET, Key=key_name, Body=PAYLOAD_1KB)
    return idx

start_time = time.time()

# Esecuzione con pool di 30 thread concorrenti per simulare centinaia di client simultanei
with ThreadPoolExecutor(max_workers=30) as executor:
    list(executor.map(upload_worker, range(1, NUM_FILE_STRESS + 1)))

elapsed = time.time() - start_time
throughput = NUM_FILE_STRESS / elapsed

print(f"\n🏆 STRESS TEST COMPLETATO CON SUCCESSO!")
print(f"  - File caricati: {NUM_FILE_STRESS} file da 1 KB")
print(f"  - Tempo impiegato: {elapsed:.2f} secondi")
print(f"  - Velocità di ingestione: {throughput:.1f} file/secondo")
print(f"  - Stato memoria NameNode/OM: STABILE (i metadati sono su RocksDB!)")

In [ ]:
# Verifichiamo il conteggio reale degli oggetti presenti nel bucket
paginator = s3.get_paginator('list_objects_v2')
conteggio_totale = 0

for page in paginator.paginate(Bucket=STRESS_BUCKET, Prefix='small_files/'):
    conteggio_totale += len(page.get('Contents', []))

print(f"📊 Conteggio verificato nel bucket '{STRESS_BUCKET}': {conteggio_totale} oggetti!")

# Mostriamo i primi 5 oggetti salvati
anteprima = s3.list_objects_v2(Bucket=STRESS_BUCKET, Prefix='small_files/', MaxKeys=5)
print("\nPrime 5 chiavi memorizzate:")
for obj in anteprima.get('Contents', []):
    print(f"  - {obj['Key']} ({obj['Size']} bytes, ETag: {obj['ETag']})")

In [ ]:
# Pulizia delle chiavi create durante lo stress test per liberare spazio
print("🧹 Avvio pulizia delle chiavi dello stress test...")

def delete_worker(idx):
    key_name = f"small_files/file_{idx:05d}.txt"
    s3.delete_object(Bucket=STRESS_BUCKET, Key=key_name)

with ThreadPoolExecutor(max_workers=30) as executor:
    list(executor.map(delete_worker, range(1, NUM_FILE_STRESS + 1)))

s3.delete_bucket(Bucket=STRESS_BUCKET)
print(f"✅ Bucket '{STRESS_BUCKET}' svuotato e rimosso con successo.")

---
# PARTE 3: 🌐 Compatibilità Nativa con l'API S3

In Apache Ozone, la compatibilità S3 non è un layer simulato o un semplice gateway di traduzione esterno: è **nativa nel cuore del sistema**.

### 🔄 Come Ozone Mappa l'API S3 nella Propria Gerarchia
Quando usi le API S3 (con Boto3, AWS CLI, Spark o Flink):
- I bucket S3 vengono creati automaticamente all'interno del volume speciale di sistema denominato **`s3v`**.
- Un bucket S3 denominato `mio-bucket` corrisponde esattamente al path Ozone `/s3v/mio-bucket`.
- Un oggetto S3 con chiave `dati/report.csv` corrisponde esattamente alla chiave Ozone `/s3v/mio-bucket/dati/report.csv`.

Verifichiamo questa duplice natura creando un bucket via S3 e ispezionandolo con i comandi nativi Ozone!

In [ ]:
DEMO_BUCKET = "lab-s3-compatibility"

# 1. Creiamo un bucket tramite client Boto3 (protocollo S3)
s3.create_bucket(Bucket=DEMO_BUCKET)
print(f"✅ Bucket '{DEMO_BUCKET}' creato via API S3!")

# 2. Carichiamo un file con metadati personalizzati
s3.put_object(
    Bucket=DEMO_BUCKET,
    Key="dataset/informazioni.json",
    Body=b'{"corso": "Big Data Architecture", "storage": "Apache Ozone", "protocollo": "Amazon S3"}',
    ContentType="application/json",
    Metadata={"autore": "studente", "ambiente": "docker-compose"}
)
print("✅ Oggetto caricato via Boto3 con metadati personalizzati!")

In [ ]:
# 3. DIMOSTRAZIONE DI COMPATIBILITA NATIVA:
# Ispezioniamo lo STESSO bucket e la STESSA chiave usando la CLI nativa di Ozone!
# Notare il prefisso /s3v/ che Ozone usa per i bucket S3.

print("🔍 Ispezione del bucket S3 tramite comando nativo 'ozone sh':")
!ozone sh bucket info /s3v/lab-s3-compatibility

print("\n🔍 Ispezione della chiave S3 tramite comando nativo 'ozone sh':")
!ozone sh key info /s3v/lab-s3-compatibility/dataset/informazioni.json

In [ ]:
# 4. Lettura del contenuto via S3 e via Ozone CLI
print("📖 Lettura via Boto3 (S3 GET):")
obj_s3 = s3.get_object(Bucket=DEMO_BUCKET, Key="dataset/informazioni.json")
print("  -> Content-Type:", obj_s3["ContentType"])
print("  -> Metadata:", obj_s3["Metadata"])
print("  -> Body:", obj_s3["Body"].read().decode("utf-8"))

print("\n📖 Lettura via CLI nativa Ozone ('ozone sh key cat'):")
!ozone sh key cat /s3v/lab-s3-compatibility/dataset/informazioni.json

In [ ]:
# 5. Pulizia del bucket dimostrativo
s3.delete_object(Bucket=DEMO_BUCKET, Key="dataset/informazioni.json")
s3.delete_bucket(Bucket=DEMO_BUCKET)
print(f"🧹 Bucket '{DEMO_BUCKET}' e oggetto rimossi con successo.")

# Pulizia del volume iniziale /volume-lab creato nella Parte 1
!ozone sh key delete /volume-lab/dati/file_benvenuto.txt 2>/dev/null || true
!ozone sh bucket delete /volume-lab/dati 2>/dev/null || true
!ozone sh volume delete /volume-lab 2>/dev/null || true
print("🧹 Volume didattico '/volume-lab' rimosso.")

---
# PARTE 4: 📊 Monitoraggio con Apache Ozone Recon Dashboard

**Recon** è il servizio di monitoraggio, diagnostica e analisi di Apache Ozone.

Puoi accedere all'interfaccia web di Recon direttamente dal tuo browser al seguente URL:
### 👉 **[http://localhost:9888](http://localhost:9888)**

#### Cosa puoi esplorare nella dashboard di Recon:
1. **Overview**: Stato di salute generale del cluster, numero di DataNode attivi e capacità di storage utilizzata/disponibile.
2. **Datanodes**: Dettagli su ciascun nodo di storage, tempo di risposta heartbeat e spazio disco.
3. **Pipelines**: Le pipeline di replica **Ratis** attive per il consenso distribuito.
4. **Containers**: I container di blocchi gestiti da SCM.
5. **OM DB Insights**: La dimensione e le statistiche dei metadati memorizzati in RocksDB nell'Ozone Manager.